In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"


# LLM Inference Workshop – Part 5
## Supervised Fine-Tuning with PEFT / LoRA

In this notebook we will:

1. Reuse the GSM8K SFT preparation utilities from `utils.py`.
2. Load a slightly larger instruct model.
3. Attach LoRA adapters with PEFT.
4. Fine-tune with `Trainer`.
5. Evaluate before and after fine-tuning.

LoRA updates only a small set of low-rank adapter weights, which greatly reduces memory use compared to full fine-tuning.



## Practical Note

If your GPU memory is limited, reduce:
- `TRAIN_SAMPLES`
- `MAX_LENGTH`
- `per_device_train_batch_size`
- `gradient_accumulation_steps`


In [ ]:
import os
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from IPython.display import Markdown

from utils import (
    evaluate_model,
    compute_accuracy,
    build_gsm8k_sft_datasets,
    SFTDataCollator,
)

In [3]:
SEED = 0

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct" # HuggingFaceTB/SmolLM2-135M-Instruct
OUTPUT_DIR = "artifacts/notebook5-sft-lora-llama-1b"

TRAIN_SAMPLES = None
EVAL_SAMPLES = None
EVAL_GENERATION_SAMPLES = 32

MAX_LENGTH = 1024
MAX_NEW_TOKENS = 512

LEARNING_RATE = 2e-5
NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
LOGGING_STEPS = 10
SAVE_STEPS = 50

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    # "gate_proj",
    # "up_proj",
    # "down_proj",
]

torch.manual_seed(SEED)

In [4]:

if torch.cuda.is_available():
    device = "cuda"
    model_dtype = "auto"
elif torch.backends.mps.is_available():
    device = "mps"
    model_dtype = torch.bfloat16
else:
    device = "cpu"
    model_dtype = torch.float32

print("Device:", device)
print("Model dtype:", model_dtype)


Device: cuda
Model dtype: auto



## Load the Tokenizer and Model

We keep caching disabled during training to reduce memory usage.


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=model_dtype,
).to(device=device)

model.config.use_cache = False
model.generation_config.pad_token_id = tokenizer.pad_token_id

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)


Tokenizer loaded.
Pad token: <|eot_id|>
EOS token: <|eot_id|>


## Gradient Checkpointing Note

When gradient checkpointing is enabled, we need to enable checkpointing on the model **before** the LoRA attachment.

In [6]:
model.gradient_checkpointing_enable()


## Attach LoRA Adapters

We insert low-rank adapters into the attention projection layers.
Only these adapters will be trainable.


In [7]:

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 6,815,744 || all params: 1,242,630,144 || trainable%: 0.5485



## Build the Training and Evaluation Datasets

The GSM8K formatting, chat templating, and label masking are moved into `utils.py`.


In [8]:

train_dataset, eval_dataset = build_gsm8k_sft_datasets(
    tokenizer=tokenizer,
    train_path="gsm8k/train.jsonl",
    eval_path="gsm8k/test.jsonl",
    train_samples=TRAIN_SAMPLES,
    eval_samples=EVAL_SAMPLES,
    seed=SEED,
    max_length=MAX_LENGTH,
)

data_collator = SFTDataCollator(pad_token_id=tokenizer.pad_token_id)

print(train_dataset)
print(eval_dataset)


Dataset({
    features: ['y_true', 'prompt_ids', 'input_ids', 'labels'],
    num_rows: 7473
})
Dataset({
    features: ['y_true', 'prompt_ids', 'input_ids', 'labels'],
    num_rows: 1319
})



## Baseline Evaluation

LoRA weights are initialized near zero, so the pre-training performance matches the base model.


In [9]:

baseline_subset, baseline_responses, baseline_metrics = evaluate_model(
    model,
    tokenizer,
    eval_dataset,
    max_new_tokens=MAX_NEW_TOKENS,
    sample_count=3,
    batch_size=EVAL_BATCH_SIZE,
)

print(
    f"Baseline accuracy on {baseline_metrics['total']} samples: {baseline_metrics['accuracy']:.4f}"
)


You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Baseline accuracy on 3 samples: 0.0000


In [10]:

print(tokenizer.decode(baseline_subset[0]["prompt_ids"]))
print()
Markdown(baseline_responses[0])


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 12 Apr 2026

You are an expert in Math.<|eot_id|><|start_header_id|>user<|end_header_id|>

Solve the following math word problem carefully.

Think step by step to ensure the reasoning is correct.
When you are confident in the final answer, output ONLY:

\boxed{FINAL_ANSWER}

Do not include any additional text in the box.

Problem:
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?<|eot_id|><|start_header_id|>assistant<|end_header_id|>





16 * 3 = 48
48 - 4 = 44
44 / 2 = 22

FINAL_ANSWER = 22


## Configure `Trainer`

We use a higher learning rate than full fine-tuning because only adapter weights are updated.


In [11]:

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,
    remove_unused_columns=False,
    gradient_checkpointing=True,
    bf16=(device != "cpu" and model_dtype == torch.bfloat16),
    fp16=(device != "cpu" and model_dtype == torch.float16),
    report_to="tensorboard",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)


/tmp/ipykernel_401806/2418362804.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
[2026-04-12 18:50:20] INFO spawn.py:77: gcc -pthread -B /nfs/home/seymol/conda_envs/v3.13/compiler_compat -fno-strict-overflow -Wsign-compare -DNDEBUG -O2 -Wall -fPIC -O2 -isystem /nfs/home/seymol/conda_envs/v3.13/include -fPIC -O2 -isystem /nfs/home/seymol/conda_envs/v3.13/include -fPIC -c /tmp/tmp7yggbtmu/test.c -o /tmp/tmp7yggbtmu/test.o
[2026-04-12 18:50:20] INFO spawn.py:77: gcc -pthread -B /nfs/home/seymol/conda_envs/v3.13/compiler_compat /tmp/tmp7yggbtmu/test.o -laio -o /tmp/tmp7yggbtmu/a.out
/nfs/home/seymol/conda_envs/v3.13/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
[2026-04-12 18:50:20] INFO spawn.py:77: gcc -pthread -B /nfs/home/seymol/conda_envs/v3.13/compiler_compat -fno-strict-overflow -Wsign-compare -DNDEBUG -O2 -Wa

[2026-04-12 18:50:20] INFO spawn.py:77: gcc -pthread -B /nfs/home/seymol/conda_envs/v3.13/compiler_compat /tmp/tmp6xb7udlb/test.o -laio -o /tmp/tmp6xb7udlb/a.out
/nfs/home/seymol/conda_envs/v3.13/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status



## Train


In [12]:

checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    checkpoints = [
        os.path.join(OUTPUT_DIR, d)
        for d in os.listdir(OUTPUT_DIR)
        if d.startswith("checkpoint-")
    ]
    if checkpoints:
        checkpoint = max(checkpoints, key=os.path.getmtime)

print("Resuming from checkpoint:", checkpoint)
train_result = trainer.train(resume_from_checkpoint=checkpoint)
train_result


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Resuming from checkpoint: None


Step,Training Loss
10,0.943800
20,0.907200
30,0.892800
40,0.771700
50,0.688500
60,0.632300
70,0.587300
80,0.571700
90,0.549500
100,0.543700


TrainOutput(global_step=1170, training_loss=0.5021754525665544, metrics={'train_runtime': 1078.6612, 'train_samples_per_second': 34.64, 'train_steps_per_second': 1.085, 'total_flos': 8.258324361279898e+16, 'train_loss': 0.5021754525665544, 'epoch': 5.0})


## Evaluate After Fine-Tuning


In [13]:

ft_subset, ft_responses, ft_metrics = evaluate_model(
    model,
    tokenizer,
    eval_dataset,
    max_new_tokens=MAX_NEW_TOKENS,
    sample_count=EVAL_GENERATION_SAMPLES,
    batch_size=EVAL_BATCH_SIZE,
)

print(
    f"Fine-tuned accuracy on {ft_metrics['total']} samples: {ft_metrics['accuracy']:.4f}"
)


Fine-tuned accuracy on 32 samples: 0.2812


In [14]:

print("Baseline accuracy:", round(baseline_metrics["accuracy"], 4))
print("Fine-tuned accuracy:", round(ft_metrics["accuracy"], 4))


Baseline accuracy: 0.0
Fine-tuned accuracy: 0.2812



## Save and Reload the LoRA Adapter

We save only the adapter weights (plus tokenizer).
To reload, we attach the adapter to a fresh base model.


In [15]:
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "lora_adapter")

In [16]:

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Saved LoRA adapter to:", ADAPTER_DIR)

reloaded_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=model_dtype,
).to(device)
reloaded_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
reloaded_model.eval()
reloaded_model.config.use_cache = True

print("Reloaded adapter from:", ADAPTER_DIR)


Saved LoRA adapter to: artifacts/notebook5-sft-lora-llama-1b/lora_adapter
Reloaded adapter from: artifacts/notebook5-sft-lora-llama-1b/lora_adapter


## vLLM Evaluation with LoRA (After Cleaning CUDA Memory)

Restart the kernel to free the memory.


### Important: Restart the Kernel Before vLLM

PyTorch can keep GPU memory fragments alive even after `del` and `empty_cache()`.
For vLLM evaluation, **restart the kernel** and run only
the minimal setup cells below.


In [3]:
# After restart: re-import minimal dependencies
import os

import torch
from transformers import AutoTokenizer

from utils import build_gsm8k_sft_datasets, compute_accuracy

from vllm import LLM, SamplingParams, TokensPrompt
from vllm.lora.request import LoRARequest

In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [5]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct" # HuggingFaceTB/SmolLM2-135M-Instruct
OUTPUT_DIR = "artifacts/notebook5-sft-lora-llama-1b"

ADAPTER_DIR = os.path.join(OUTPUT_DIR, "lora_adapter")

SEED = 0
MAX_LENGTH = 1024
MAX_NEW_TOKENS = 512
LORA_R = 32

train_dataset, eval_dataset = build_gsm8k_sft_datasets(
    tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME),
    train_path="gsm8k/train.jsonl",
    eval_path="gsm8k/test.jsonl",
    seed=SEED,
    max_length=MAX_LENGTH,
)

In [6]:
# Build prompts from the tokenized dataset
prompt_tokens = [TokensPrompt(prompt_token_ids=x) for x in eval_dataset["prompt_ids"]]

llm = LLM(
    model=MODEL_NAME,
    max_model_len=MAX_LENGTH + MAX_NEW_TOKENS,
    enforce_eager=True,
    gpu_memory_utilization=0.8,
    enable_lora=True,  ## Use LoRA
    max_lora_rank=LORA_R,
)

sampling_params = SamplingParams(
    max_tokens=MAX_NEW_TOKENS,
    temperature=0.0,
)

lora_request = LoRARequest("lora_adapter", 1, ADAPTER_DIR)
outputs = llm.generate(prompt_tokens, sampling_params, lora_request=lora_request)
vllm_responses = [out.outputs[0].text for out in outputs]

vllm_metrics = compute_accuracy(responses=vllm_responses, dataset=eval_dataset)
print(
    f"vLLM accuracy on {vllm_metrics['total']} samples: {vllm_metrics['accuracy']:.4f}"
)

INFO 04-12 19:37:34 [utils.py:253] non-default args: {'seed': None, 'max_model_len': 1536, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'enforce_eager': True, 'enable_lora': True, 'max_lora_rank': 32, 'model': 'meta-llama/Llama-3.2-1B-Instruct'}
WARNING 04-12 19:37:34 [arg_utils.py:1175] `seed=None` is equivalent to `seed=0` in V1 Engine. You will no longer be allowed to pass `None` in v0.13.
INFO 04-12 19:37:35 [model.py:637] Resolved architecture: LlamaForCausalLM
INFO 04-12 19:37:35 [model.py:1750] Using max model len 1536
INFO 04-12 19:37:35 [scheduler.py:228] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 04-12 19:37:35 [vllm.py:601] Enforce eager set, overriding optimization level to -O0
INFO 04-12 19:37:35 [vllm.py:707] Cudagraph is disabled under eager mode
(EngineCore_DP0 pid=531949) INFO 04-12 19:37:36 [core.py:93] Initializing a V1 LLM engine (v0.12.0) with config: model='meta-llama/Llama-3.2-1B-Instruct', speculative_config=None, tokenizer

(EngineCore_DP0 pid=531949) /nfs/home/seymol/conda_envs/v3.13/lib/python3.13/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:181: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=531949) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=531949)   warnings.warn(


(EngineCore_DP0 pid=531949) INFO 04-12 19:37:48 [cuda.py:411] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION']
(EngineCore_DP0 pid=531949) INFO 04-12 19:37:49 [weight_utils.py:527] No model.safetensors.index.json found in remote.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore_DP0 pid=531949) INFO 04-12 19:37:49 [default_loader.py:308] Loading weights took 0.63 seconds
(EngineCore_DP0 pid=531949) INFO 04-12 19:37:49 [punica_selector.py:20] Using PunicaWrapperGPU.
(EngineCore_DP0 pid=531949) INFO 04-12 19:37:52 [gpu_model_runner.py:3549] Model loading took 2.3765 GiB memory and 6.601017 seconds
(EngineCore_DP0 pid=531949) INFO 04-12 19:37:56 [gpu_worker.py:359] Available KV cache memory: 27.98 GiB
(EngineCore_DP0 pid=531949) INFO 04-12 19:37:56 [kv_cache_utils.py:1286] GPU KV cache size: 916,800 tokens
(EngineCore_DP0 pid=531949) INFO 04-12 19:37:56 [kv_cache_utils.py:1291] Maximum concurrency for 1,536 tokens per request: 596.88x
(EngineCore_DP0 pid=531949) INFO 04-12 19:37:56 [core.py:254] init engine (profile, create kv cache, warmup model) took 4.28 seconds
(EngineCore_DP0 pid=531949) WARNING 04-12 19:37:58 [vllm.py:608] Inductor compilation was disabled by user settings,Optimizations settings that are only active duringInductor compilation wi

Adding requests:   0%|          | 0/1319 [00:00<?, ?it/s]

WARNING 04-12 19:37:58 [input_processor.py:243] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|          | 0/1319 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

(EngineCore_DP0 pid=531949) WARNING 04-12 19:37:59 [utils.py:250] Using default LoRA kernel configs
vLLM accuracy on 1319 samples: 0.4003


In [7]:
print(vllm_responses[0])

She eats 3 eggs for breakfast every morning, so she has 16 - 3 = <<16-3=13>>13 eggs left.
She bakes muffins for her friends every day, so she has 13 - 4 = <<13-4=9>>9 eggs left.
She sells the 9 eggs at the farmers' market for $2 each, so she makes 9 * 2 = $<<9*2=18>>18 every day.
\boxed{18}



## Summary

In this notebook we:
- built GSM8K SFT datasets with helper functions in `utils.py`
- attached LoRA adapters with PEFT
- fine-tuned using `Trainer`
- evaluated the fine-tuned model by generation

In the next step, we can scale training across multiple GPUs with Accelerate or DeepSpeed.
